In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

sns.set_theme(style="whitegrid")

# ==========================================
# 1. LOAD DATA & DEFINE STRATIFIED LOBO SPLIT
# ==========================================
df = pd.read_csv('/kaggle/input/YOUR_DATASET_PATH/ML_Engineered_Battery_Dataset.csv')

# The 9 test batteries (1 from each condition group)
test_batteries = ['B0005', 'B0025', 'B0029', 'B0033', 'B0038', 'B0041', 'B0045', 'B0049', 'B0053']

# Split data: If Battery_ID is in the list -> Test Set. Otherwise -> Train Set.
df_train = df[~df['Battery_ID'].isin(test_batteries)].copy()
df_test = df[df['Battery_ID'].isin(test_batteries)].copy()

# Features (X) and Target (y)
features = [
    'Cycle_Index', 'Ambient_Temperature', 'Discharge_Time_Seconds', 
    'Max_Temp_Reached', 'Min_Voltage_Recorded', 
    'Voltage_Drop_Rate_V_per_sec', 'Internal_Resistance_Re'
]
target = 'SoH'

X_train, y_train = df_train[features], df_train[target]
X_test, y_test = df_test[features], df_test[target]

print(f"Training on {len(df_train['Battery_ID'].unique())} batteries ({len(X_train)} cycles)")
print(f"Testing on {len(df_test['Battery_ID'].unique())} unseen batteries ({len(X_test)} cycles)")

# ==========================================
# 2. TRAIN THE MODELS
# ==========================================
print("\nTraining Track A Models...")

# Model 1: Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Model 2: XGBoost (Using Kaggle GPU)
xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, tree_method='hist', device='cuda', random_state=42)
xgb_model.fit(X_train, y_train)

# Model 3: LightGBM (CPU is extremely fast, but can use device_type='gpu' if configured)
lgb_model = LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
lgb_model.fit(X_train, y_train)

# ==========================================
# 3. BENCHMARK COMPARISON ON UNSEEN BATTERY
# ==========================================
# Let's test them on Battery B0005 (Room Temp, 2A)
target_test_battery = 'B0005'
df_b0005 = df_test[df_test['Battery_ID'] == target_test_battery].copy()
X_b0005 = df_b0005[features]

# Generate Predictions
df_b0005['Pred_RF'] = rf_model.predict(X_b0005)
df_b0005['Pred_XGB'] = xgb_model.predict(X_b0005)
df_b0005['Pred_LGBM'] = lgb_model.predict(X_b0005)

# Calculate RMSE for each model
rmse_rf = root_mean_squared_error(df_b0005['SoH'], df_b0005['Pred_RF'])
rmse_xgb = root_mean_squared_error(df_b0005['SoH'], df_b0005['Pred_XGB'])
rmse_lgbm = root_mean_squared_error(df_b0005['SoH'], df_b0005['Pred_LGBM'])

print(f"\n--- RMSE on Unseen Battery {target_test_battery} ---")
print(f"Random Forest: {rmse_rf:.4f}")
print(f"XGBoost:       {rmse_xgb:.4f}")
print(f"LightGBM:      {rmse_lgbm:.4f}")

# Plotting the Results
plt.figure(figsize=(14, 7))
plt.plot(df_b0005['Cycle_Index'], df_b0005['SoH'], label='Actual SoH (Ground Truth)', color='black', linewidth=3)
plt.plot(df_b0005['Cycle_Index'], df_b0005['Pred_RF'], label=f'Random Forest (RMSE: {rmse_rf:.3f})', linestyle='--')
plt.plot(df_b0005['Cycle_Index'], df_b0005['Pred_XGB'], label=f'XGBoost (RMSE: {rmse_xgb:.3f})', linestyle='-.')
plt.plot(df_b0005['Cycle_Index'], df_b0005['Pred_LGBM'], label=f'LightGBM (RMSE: {rmse_lgbm:.3f})', linestyle=':')

# Draw the Universal Dead Barrier (70% SoH)
plt.axhline(y=0.70, color='red', alpha=0.5, label='End of Life Threshold (70%)')

plt.title(f"Track A Benchmark: SoH Prediction on Completely Unseen Battery ({target_test_battery})")
plt.xlabel("Cycle Index")
plt.ylabel("State of Health (SoH)")
plt.legend()
plt.tight_layout()
plt.show()
